In [1]:
import pandas as pd
import sqlite3
import json

# 1. Extracting the Nested JSON (Customer Profiles)
def load_customer_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    parsed_customers = []
    # The JSON keys are formatted like "usr_profile_CUST-0001"
    for key, val in data.items():
        customer_id = key.replace('usr_profile_', '')
        
        # Flattening the nested dictionaries
        row = {
            'customer_id': customer_id,
            'macro_region': val.get('account_details', {}).get('geo_segmentation', {}).get('macro_region'),
            'primary_goal': val.get('account_details', {}).get('preferences', {}).get('primary_goal'),
            'utm_medium': val.get('acquisition_telemetry', {}).get('source', {}).get('utm_medium'),
            'primary_device': val.get('acquisition_telemetry', {}).get('hardware', {}).get('primary_device'),
            'tier': val.get('financial_metrics_v2', {}).get('status', {}).get('tier'),
            'reward_points': val.get('financial_metrics_v2', {}).get('status', {}).get('reward_points'),
            'return_rate_pct': val.get('financial_metrics_v2', {}).get('risk_factors', {}).get('return_rate_pct')
        }
        parsed_customers.append(row)
        
    return pd.DataFrame(parsed_customers)

# 2. Extracting the SQLite Database (Transactions/Orders)
def load_transaction_data(sqlite_path):
    conn = sqlite3.connect(sqlite_path)
    
    # First, let's see what tables exist (You can run this once to check)
    # tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
    # print(tables)
    
    # Assuming the main table is named 'orders' or 'transactions' based on standard conventions
    # You will need to adjust 'transactions' to the actual table name you find
    query = "SELECT * FROM transactions" # Change 'transactions' based on your DB schema
    df_transactions = pd.read_sql_query(query, conn)
    conn.close()
    return df_transactions

# 3. Extracting the Mystery .sys file (System Logs)
# Looking at the snippet, it's delimited by tildes (~)
def load_sys_logs(sys_path):
    # Defining column names based on the data footprint
    columns = ['YearMonth', 'Manager', 'LaborCost', 'WarehouseRent', 'Metric_1', 'Metric_2', 'Rate', 'Score', 'Temp']
    df_sys = pd.read_csv(sys_path, sep='~', names=columns)
    return df_sys

In [2]:
def transform_data(df_customers, df_transactions):
    # Clean column names (standardize to lowercase_with_underscores)
    df_transactions.columns = [c.lower().replace(' ', '_') for c in df_transactions.columns]
    
    # Merge Transactions with Customer Profiles
    # Assuming the transaction table has a 'customer_id' or 'cust_id' column
    # Let's assume it's 'customer_id' for this example
    df_merged = pd.merge(df_transactions, df_customers, on='customer_id', how='left')
    
    # Handle missing values & format dates
    # Assuming a 'date' or 'order_date' column exists in transactions
    if 'order_date' in df_merged.columns:
        df_merged['order_date'] = pd.to_datetime(df_merged['order_date'])
        df_merged['order_month'] = df_merged['order_date'].dt.to_period('M').astype(str)
        
    return df_merged

In [3]:
def generate_business_outputs(df_merged):
    # Output 1: Single Source of Truth (Analytics Database)
    engine = sqlite3.connect('zenith_analytics_warehouse.db')
    df_merged.to_sql('fact_orders_enriched', engine, if_exists='replace', index=False)
    
    # Output 2: Customers per month by acquisition channel
    if 'order_month' in df_merged.columns and 'utm_medium' in df_merged.columns:
        acq_dataset = df_merged.groupby(['order_month', 'utm_medium'])['customer_id'].nunique().reset_index()
        acq_dataset.to_csv('customers_by_channel.csv', index=False)
    
    # Output 3: Revenue/Orders per SKU (Assuming 'sku' and 'revenue' columns exist)
    if 'sku' in df_merged.columns and 'revenue' in df_merged.columns:
        sku_dataset = df_merged.groupby('sku').agg({'customer_id':'count', 'revenue':'sum'}).reset_index()
        sku_dataset.rename(columns={'customer_id': 'total_orders'}, inplace=True)
        sku_dataset.to_csv('revenue_per_sku.csv', index=False)
        
    print("Pipeline Execution Complete. Warehouse populated and reports generated.")

# --- RUNNING THE PIPELINE ---
df_cust = load_customer_data('customer_profiles_v2.json')
df_trans = load_transaction_data('zenith_core_db.sqlite')
# df_sys = load_sys_logs('sys_ops_log_archive.sys') # Keep ready for ops analysis

df_final = transform_data(df_cust, df_trans)
generate_business_outputs(df_final)

FileNotFoundError: [Errno 2] No such file or directory: 'customer_profiles_v2.json'